- nano-banana based：不可编辑
    - 多图一致的实现：agent loop based
        - https://github.com/op7418/NanoBanana-PPT-Skills
        - 先 llm 生成 slides_plan.json
            - 每页写什么、是什么页型
                - page type（页型）：封面/内容/数据
- codex ppt skills
    - https://github.com/openai/skills/commit/fdf90d652aea00d3fa57803a348744b1d7670fcb#diff-cb4b5455ee8e0da62d9f8ccb870743fa6e8ce85764090cd2179aa4c3e313b4bf
        - PptxGenJS => pptx
- 复刻 ppt
    - 风格：布局，色彩，文字和图片的组合；

### 可编辑的 pptx

- https://github.com/dnnyngyen/kimi-agent-internals/blob/main/prompts/slides.md
    - .pptx.html 是“用 HTML/CSS 描述每一页幻灯片”的中间表示，最终由 slides_generator 转成 PowerPoint 文件。
    - generate_slides_outline：负责“把需求/资料/搜索结果→结构化大纲”，并强制走用户确认与修改回传。
    - slides_generator：负责“把 HTML 版逐页源码写入 .pptx.html → 转为 .pptx”。
        - https://github.com/dnnyngyen/kimi-agent-internals/blob/main/tools/ok-computer/slides_generator.md
- 并不是直接编辑 .pptx 二进制文件，而是通过 "HTML 中间层 + 格式转换引擎" 的技术路径来实现 PPT 创作的。具体原理如下：
    - 需求 → HTML/CSS/JS 代码 → 转换引擎 → .pptx 文件
    - Web 技术（HTML5 + CSS3 + JavaScript）来构建演示文稿：
        - 结构：用 HTML 的 div 元素表示每一页幻灯片（`<div class="ppt-slide">`）
        - 样式：用 Tailwind CSS 定义布局、颜色、字体、间距等视觉效果
        - 交互：用 Echarts、Chart.js 等库绘制数据图表
    - slides_generator (kimi)：
        - HTML 渲染引擎：解析生成的 HTML/CSS
        - PPTX 转换器：将渲染后的 DOM 结构映射为 Office Open XML 格式（.pptx 的标准格式）
        - 资源处理器：处理图片、字体等嵌入资源
    - 格式转换原理
        - 将 HTML 的 div 转换为 PPT 的 Slide 对象
        - 将 CSS 样式（颜色、字体、边距）转换为 PPT 的 Shape 属性
        - 将 SVG/Canvas 图表转换为 PPT 中的矢量图形或图片
        - 最终打包成符合 OOXML 标准的 .pptx 文件

```mermaid
sequenceDiagram
  autonumber
  actor U as 用户
  participant A as Slides Agent
  participant FR as 文件读取器
  participant VS as 视觉方案/模板解析
  participant WS as 信息检索(网页/数据API)
  participant IS as 图片检索(插图)
  participant OT as 大纲工具(generate_slides_outline)
  participant SG as 生成器(slides_generator)
  participant OUT as 输出目录(/mnt/okcomputer/output)

  Note over A: 页数约束：最多 30 页；若用户未指定页数，默认控制在 ≤12 页
  Note over A: 有模板：页类型来自 supported_page_types；无模板：cover/table_of_contents/chapter/content/final（不提供 thanks）

  U->>A: 提供需求(主题/受众/目的/页数偏好)\n+ 资料(可上传文件)\n+ 可选：模板风格JSON
  A->>FR: 读取并解析上传文件
  FR-->>A: 返回可用内容(文本/图片/表格/要点)

  alt 用户选择了模板
    A->>VS: 解析模板风格JSON(配色/字体/可用页类型等)
    VS-->>A: 返回模板约束与样式参数
  else 用户未选择模板
    A->>VS: 生成视觉方案文档(写入Markdown)\n(统一配色/字体/版式节奏)
    VS-->>A: 返回方案文档路径/样式参数
  end

  opt 需要补全信息 且 用户未禁止搜索
    A->>WS: 两阶段检索：广搜→深挖(网页/论文/数据源API)
    WS-->>A: 返回补全事实/数据/参考信息
  end

  A->>OT: 生成初稿大纲\n(JSON: number/type/title/content)
  OT-->>U: 展示大纲并请求确认/修改

  loop 用户迭代修改直到确认
    U-->>OT: 修改大纲(增删页/改标题/改正文/改页型/调顺序)
    OT-->>U: 返回更新后的大纲供继续调整
  end

  U-->>OT: 最终确认提交
  OT-->>A: 返回最终大纲(JSON)

  A->>IS: 检索插图(仅插图，不含图表/流程图)
  IS-->>A: 返回图片素材(链接/元信息)

  A->>A: 生成图表/流程图/SmartArt\n(程序绘制或排版生成，不依赖图片搜索)

  A->>SG: 生成 .pptx.html 源文件\n(逐页HTML + ppt-slide CSS)\n并触发转换为 .pptx
  SG->>OUT: 写入 xxx.pptx.html 与 xxx.pptx
  OUT-->>U: 交付/下载 PPTX
```

### ppt 风格

- styles/vector-illustration.md: 矢量插画风格
- styles/gradient-glass.md: 渐变拟物玻璃卡片风格
    - Bento 网格